In [1]:
!pip install -q -U langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install sentence-transformers

In [3]:
!pip -q install langchain-huggingface  langchain-text-splitters faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.8 MB/s eta 0:00:00


In [4]:
!pip install -q yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.3 MB/s eta 0:00:00


In [5]:
import yt_dlp

print(yt_dlp.version.__version__)

2026.06.09


In [8]:
import yt_dlp

url = 'https://www.youtube.com/watch?v=aircAruvnKk'
ydl_opts  = {
    'format': 'bestaudio/best',
    "outtmpl": "audio.%(ext)s",
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
  ydl.download([url])

[youtube] Extracting URL: https://www.youtube.com/watch?v=aircAruvnKk
[youtube] aircAruvnKk: Downloading webpage


[youtube] aircAruvnKk: Downloading android vr player API JSON
[info] aircAruvnKk: Downloading 1 format(s): 251
[download] Destination: audio.webm
[download] 100% of   17.92MiB in 00:00:00 at 40.27MiB/s  


In [9]:
!pip install -q faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 52.3 MB/s eta 0:00:00


In [11]:
from faster_whisper import WhisperModel

print("Whisper imported successfully")

Whisper imported successfully


In [13]:
from faster_whisper import WhisperModel
model = WhisperModel(
    'base',
    device = 'cpu',
    compute_type= 'int8'
)
print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model loaded successfully


In [15]:
segments, info = model.transcribe(
    'audio.webm',
    beam_size=5
)
full_text = ""
for segment in segments:
  full_text += segment.text + ""

print(full_text[:1000])

 This is a 3. It's sloppily written and rendered at an extremely low resolution of 28 x 28 pixels, but your brain has no trouble recognizing it as a 3. And I want you to take a moment to appreciate how crazy it is that brains can do this so effortlessly. I mean this, this, and this are also recognizable as 3s, even though the specific values of each pixel is very different from one image to the next. The particular light-sensitive cells in your eye that are firing when you see this 3 are very different from the ones firing when you see this 3. But something in that crazy smart visual cortex of yours resolves these as representing the same idea, while at the same time recognizing other images as their own distinct ideas. But if I told you, hey, sit down and write for me a program that takes in a grid of 28 x 28 pixels like this and outputs a single number between 0 and 10, telling you what it thinks the digit is, while the task goes from comically trivial to dauntingly difficult. Unless

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunk = text_splitter.split_text(full_text)
print('total chunks', len(chunk))
print(chunk[0])

total chunks 41
This is a 3. It's sloppily written and rendered at an extremely low resolution of 28 x 28 pixels, but your brain has no trouble recognizing it as a 3. And I want you to take a moment to appreciate how crazy it is that brains can do this so effortlessly. I mean this, this, and this are also recognizable as 3s, even though the specific values of each pixel is very different from one image to the next. The particular light-sensitive cells in your eye that are firing when you see this 3 are very


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings
embed_model = HuggingFaceEmbeddings( model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully


In [18]:
from langchain_community.vectorstores import FAISS

# FAISS Vector Store
vectorstore = FAISS.from_texts(
    texts=chunk,
    embedding=embed_model
)

print("FAISS Vector Store created successfully")

/tmp/ipykernel_580/1297858633.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


FAISS Vector Store created successfully


In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})
print("Retriever created successfully")

Retriever created successfully


In [20]:
from transformers import pipeline

# TinyLlama Pipeline
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=256
)

print("TinyLlama loaded successfully")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


TinyLlama loaded successfully


In [22]:
while True:
    question = input("\nAsk a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Create context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Professional prompt
    prompt = f"""
You are a helpful AI assistant.

Use only the provided context to answer the user's question.

If the answer is not present in the context, simply say:
"I don't know based on the provided context."

Keep the answer clear, concise, and natural.

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer
    response = pipe(prompt)

    generated_text = response[0]["generated_text"]

    # Extract only the answer part
    answer = generated_text.split("Answer:")[-1].strip()

    print("\nAnswer:")
    print(answer)


Ask a question (type 'exit' to quit): What is a neural network

Answer:
a neural network is an artificial neural system that mimics the functioning of the human brain. It is a type of artificial neural network.

Ask a question (type 'exit' to quit): exit
